In [2]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [3]:


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [14]:
fecha_ref='2026-05-01'
# engine_mysql.dispose()

query = f"""
SELECT
    cid,
    telefono AS phone_number,
    fecha_llamada,
    hora_llamada,
    duracion,

    CASE
        WHEN tipificacion = 'AB'   THEN 'OCUPADO'
        WHEN tipificacion = 'NA'   THEN 'LINEA SATURADA'
        WHEN tipificacion = 'AA'   THEN 'TELEFONO APAGADO'
        WHEN tipificacion = 'DROP' THEN 'Agente no disponible'
        WHEN tipificacion = 'ADC'  THEN 'TELEFONO APAGADO'
    END AS nombre,

    CASE
        WHEN tipificacion = 'AB'   THEN 25
        WHEN tipificacion = 'NA'   THEN 26
        WHEN tipificacion = 'AA'   THEN 25
        WHEN tipificacion = 'DROP' THEN 27
        WHEN tipificacion = 'ADC'  THEN 25
    END AS id_banco,

    respuesta AS call_result,
    tipificacion as codigo,
    null as fecha_agenda,
    campana

FROM crm_target.valentina_llamadas
WHERE app = 17
  AND fecha_llamada >= '{fecha_ref}'
"""

df_maquina = pd.read_sql(query, engine_mysql)
query = f"""
SELECT
    a.ll_cid AS cid,
    a.ll_numero AS phone_number,
    a.ll_fecha AS fecha_llamada,
    a.ll_ini_registro AS hora_llamada,
    a.ll_duracion AS duracion,

    b.nombre,
    b.id_banco,
    null as call_result,
    b.codigo,
    a.ll_fecha_llamar AS fecha_agenda,
    a.campana

FROM crm_target.alfcc_llamadas a
LEFT JOIN crm_target.alfcc_acciones b
    ON a.ll_accion = b.id
WHERE a.ll_base = 'mayo 2026'
  AND a.ll_fecha >= '{fecha_ref}'
"""

df_agente = pd.read_sql(query, engine_mysql)

df_vicidial = pd.concat([df_maquina, df_agente], ignore_index=True)


C:\Users\DATA\AppData\Local\Temp\ipykernel_14660\2643167968.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_vicidial = pd.concat([df_maquina, df_agente], ignore_index=True)


In [5]:
df_vicidial.head()

,cid,phone_number,fecha_llamada,hora_llamada,duracion,nombre,id_banco,call_result,codigo,fecha_agenda,campana
0,3445167,987340962,2026-05-11,0 days 09:00:18,0,OCUPADO,25,Busy Auto,AB,NaT,ALFCC
1,3526996,960149207,2026-05-11,0 days 09:00:18,0,OCUPADO,25,Busy Auto,AB,NaT,ALFCC
2,3488247,999339050,2026-05-11,0 days 09:00:57,0,LINEA SATURADA,26,No Answer AutoDial,NA,NaT,ALFCC
3,3440406,945227771,2026-05-11,0 days 09:00:40,0,TELEFONO APAGADO,25,Answering Machine Auto,AA,NaT,ALFCC
4,3435875,994455341,2026-05-11,0 days 09:00:18,0,OCUPADO,25,Busy Auto,AB,NaT,ALFCC


In [15]:
query = f"""
    SELECT 
    nombre as descripcion,
    id_banco,
    id as peso
    FROM VALENTINA.dbo.alfcc_tipificaciones
    where estado='a'
    """
df_tipi=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)
df_tipi=df_tipi.toPandas()

In [ ]:
df_tipi[df_tipi['descripcion'].contain('tasa')].head()

,descripcion,id_banco,peso
0,CLIENTE CONFIRMA QUE YA DESEMBOLSÓ,1,1
1,CLIENTE ACEPTA OFERTA Y PASARÁ A PROCESO DE DE...,2,2
2,CITA AGENDADA,3,3
3,SEGUIMIENTO – LLAMAR CON AJUSTE DE TASA/MONTO/...,4,4
4,SEGUIMIENTO – VOLVER A LLAMAR CON MISMA OFERTA,5,5


In [11]:
df_tipi['id_banco'] = pd.to_numeric(
    df_tipi['id_banco'],
    errors='coerce'
)
df_vicidial = df_vicidial.merge(
    df_tipi,
    on='id_banco',
    how='inner'
)

In [16]:

ruta_archivo = os.path.join(ruta_csv, 'tipificaicones_alfcc_2.xlsx')
df_vicidial.to_excel(ruta_archivo, index=False)

In [ ]:

ruta_archivo = os.path.join(ruta_csv, 'tipificaicones_alfcc_2.csv')
df_vicidial.to_excel(ruta_archivo, index=False)

In [ ]:
df_vicidial=df_vicidial.join(df_tipi,["codigo"],"left")


In [ ]:
name_base='mayo 2026'
app_campana=17
cero=1
def vicidial_hoy_valentina(name_campana,name_base,app_campana,cero):
    query = f"""
    SELECT
        cid,
        telefono AS contacto,
        fecha_llamada,
        duracion,
        CASE
            WHEN tipificacion = 'AB'   THEN 25
            WHEN tipificacion = 'NA'   THEN 26
            WHEN tipificacion = 'AA'   THEN 25
            WHEN tipificacion = 'DROP' THEN 27
            WHEN tipificacion = 'ADC'  THEN 25
        END AS codigo,
        null as dni_ejecutivo
    FROM crm_target.valentina_llamadas
    WHERE app = {app_campana}
    AND fecha_llamada = CURDATE()- INTERVAL {cero} DAY
    """

    df_maquina = pd.read_sql(query, engine_mysql)

    query = f"""
    SELECT
        a.ll_cid AS cid,
        a.ll_numero AS contacto,
        a.ll_fecha AS fecha_llamada,
        a.ll_duracion AS duracion,
        b.id_banco AS codigo,
        CAST(NULL AS CHAR) AS dni_ejecutivo
    FROM crm_target.{name_campana}_llamadas a
    LEFT JOIN crm_target.{name_campana}_acciones b
        ON a.ll_accion = b.id
    WHERE a.ll_base = '{name_base}'
    AND a.ll_fecha = CURDATE()- INTERVAL {cero} DAY
    """

    df_agente = pd.read_sql(query, engine_mysql)

    df = pd.concat([df_maquina, df_agente], ignore_index=True)
    ruta_archivo = os.path.join(ruta_csv, 'tmp_vici.csv')
    df.to_csv(ruta_archivo, index=False,sep=';')

vicidial_hoy_valentina(name_base,app_campana,cero)

In [87]:
df_vicidial_hoy.head()

,cid,contacto,fecha_llamada,duracion,codigo,dni_ejecutivo
0,3445167,987340962,2026-05-11,0,25,None
1,3526996,960149207,2026-05-11,0,25,None
2,3488247,999339050,2026-05-11,0,26,None
3,3440406,945227771,2026-05-11,0,25,None
4,3435875,994455341,2026-05-11,0,25,None


In [90]:
df_vicidial_hoy_spark.show()

Py4JJavaError: An error occurred while calling o68.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (DESKTOP-47B8JAC executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:281)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:309)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:72)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoin

In [80]:
df_vicidial.head()

,cid,contacto,fecha_llamada,duracion,codigo,dni_ejecutivo
0,3590300,973940402,2026-05-05,0,25,None
1,3577362,939292432,2026-05-05,5,27,None
2,3539428,918211288,2026-05-05,0,25,None
3,3576667,997793429,2026-05-05,0,26,None
4,3576472,979666608,2026-05-05,5,27,None


In [ ]:
    query = f"""
        select 
        dni_cliente,
        fecha as fecha_llamada,Celular as contacto,
        Tipificacion as codigo,
        tmo as duracion,
        dni_ejecutivo
        from VALENTINA.dbo.{tb_gestiones}
        WHERE CAST(fecha AS DATE) >= CAST('{fecha_mes_base}' AS DATE)
        AND CAST(fecha AS DATE) < DATEADD(MONTH, 1, CAST('{fecha_mes_base}' AS DATE))
        """
    df_vicidial=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)


In [75]:
query = f"""
SELECT *
FROM crm_target.alfcc_feedback
where fecha_llamada='2026-05-05'
"""

df_feedback = pd.read_sql(query, engine_mysql)


In [78]:
df_feedback.head()

,id,NUMDOC,PERIODO,FECHA_LLAMADA,HORA_LLAMADA,HORA_TERMINO,TELEFONO,CODTIPIF,IDASESOR,TMO,tipo
0,13811898,29299447,202605,2026-05-05,0 days 13:57:16,0 days 13:57:26,964417676,22,007429793,10,REAL
1,13811897,05223025,202605,2026-05-05,0 days 13:56:42,0 days 13:57:12,998832944,10,007429793,30,REAL
2,13811896,02810625,202605,2026-05-05,0 days 13:05:00,0 days 13:05:48,969410238,5,75416659,48,REAL
3,13811895,07525259,202605,2026-05-05,0 days 13:00:00,0 days 13:04:32,950000861,2,75416659,272,REAL
4,13811894,42025683,202605,2026-05-05,0 days 13:00:09,0 days 13:02:46,929834099,5,74689231,157,REAL


In [76]:
df_vicidial.count()


cid              15923
phone_number     15923
fecha_llamada    15923
hora_llamada     15923
duracion         15923
nombre           15923
id_banco         15923
call_result      14361
codigo           15923
fecha_agenda        28
campana          15923
dtype: int64

In [77]:
df_feedback.count()

id               15973
NUMDOC           15973
PERIODO          15973
FECHA_LLAMADA    15973
HORA_LLAMADA     15973
HORA_TERMINO     15973
TELEFONO         15973
CODTIPIF         15973
IDASESOR         15973
TMO              15973
tipo             15973
dtype: int64

In [ ]:
df_feedback.head()

,id,NUMDOC,PERIODO,FECHA_LLAMADA,HORA_LLAMADA,HORA_TERMINO,TELEFONO,CODTIPIF,IDASESOR,TMO,tipo
0,1,42617046,202509,2025-09-02,0 days 09:53:30,0 days 09:53:30,942484175,50,99999999,0,VAL
1,2,73934013,202509,2025-09-02,0 days 09:53:46,0 days 09:53:46,952965860,50,99999999,0,VAL
2,3,47094300,202509,2025-09-02,0 days 09:53:48,0 days 09:53:54,959638267,50,99999999,5,VAL
3,4,46964786,202509,2025-09-02,0 days 09:53:56,0 days 09:53:56,969251955,50,99999999,0,VAL
4,5,71634853,202509,2025-09-02,0 days 09:54:10,0 days 09:54:10,906765374,50,99999999,0,VAL


In [ ]:

print(df_maquina.columns.tolist())
print(df_agente.columns.tolist())

['cid', 'phone_number', 'fecha_llamada', 'hora_llamada', 'duracion', 'nombre', 'id_banco', 'call_result', 'codigo', 'fecha_agenda', 'campana']
['cid', 'phone_number', 'fecha_llamada', 'hora_llamada', 'duracion', 'nombre', 'id_banco', 'call_result', 'codigo', 'fecha_agenda', 'campana']


ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MariaDB server version for the right syntax to use near 'tipificacion = 'ADC'  THEN 25\n    END AS id_banco,\nrespuesta as call_result\n,...' at line 15")
[SQL: 
select
cid, telefono as phone_number, fecha_llamada, hora_llamada, duracion,
CASE
    WHEN tipificacion = 'AB'   THEN 'OCUPADO'
    WHEN tipificacion = 'NA'   THEN 'LINEA SATURADA'
    WHEN tipificacion = 'AA'   THEN 'TELEFONO APAGADO'
    WHEN tipificacion = 'DROP' THEN 'Agente no disponible'
    WHEN tipificacion = 'ADC'  THEN 'TELEFONO APAGADO'
END AS nombre,
CASE
    WHEN tipificacion = 'AB'   THEN 25
    WHEN tipificacion = 'NA'   THEN 26
    WHEN tipificacion = 'AA'   THEN 25
    WHEN tipificacion = 'DROP' THEN 27
    tipificacion = 'ADC'  THEN 25
    END AS id_banco,
respuesta as call_result
, tipificacion, respuesta as call_result, campana
from crm_target.valentina_llamadas
where app=

In [ ]:

query = """
select
cid, telefono as phone_number, fecha_llamada, hora_llamada, duracion,
CASE
    WHEN tipificacion = 'AB'   THEN 'OCUPADO'
    WHEN tipificacion = 'NA'   THEN 'LINEA SATURADA'
    WHEN tipificacion = 'AA'   THEN 'TELEFONO APAGADO'
    WHEN tipificacion = 'DROP' THEN 'Agente no disponible'
    WHEN tipificacion = 'ADC'  THEN 'TELEFONO APAGADO'
END AS nombre,
CASE
    WHEN tipificacion = 'AB'   THEN 25
    WHEN tipificacion = 'NA'   THEN 26
    WHEN tipificacion = 'AA'   THEN 25
    WHEN tipificacion = 'DROP' THEN 27
    tipificacion = 'ADC'  THEN 25
    END AS id_banco,
respuesta as call_result
, tipificacion, respuesta as call_result, campana
from crm_target.valentina_llamadas
where app=17
and fecha_llamada >='2026-05-01'
AND fecha_llamada <= LAST_DAY('2026-05-01')
limit 10
"""
df_maquina = pd.read_sql(query, engine_mysql)


OperationalError: (pymysql.err.OperationalError) (1054, "Unknown column 'b.codi' in 'SELECT'")
[SQL: 
SELECT
    a.ll_cid AS cid,
    a.ll_numero AS phone_number,
    a.ll_fecha AS fecha_llamada,
    a.ll_ini_registro AS hora_llamada,
    a.ll_duracion AS duracion,
    b.nombre,
    b.id_banco,
    b.codi,
    a.ll_hora AS tramo,
    a.ll_fecha_llamar AS fecha_agenda,
    a.campana
FROM crm_target.alfcc_llamadas a
LEFT JOIN crm_target.alfcc_acciones b
    ON a.ll_accion = b.id
WHERE a.ll_base = 'mayo 2026'
  AND a.ll_fecha = '2026-05-11'
LIMIT 10
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

OperationalError: (pymysql.err.OperationalError) (1054, "Unknown column 'b.codi' in 'SELECT'")
[SQL: 
SELECT
    a.ll_cid AS cid,
    a.ll_numero AS phone_number,
    a.ll_fecha AS fecha_llamada,
    a.ll_ini_registro AS hora_llamada,
    a.ll_duracion AS duracion,

    b.nombre,
    b.id_banco,
    b.codi,

    a.ll_hora AS tramo,
    a.ll_fecha_llamar AS fecha_agenda,
    a.campana

FROM crm_target.alfcc_llamadas a
LEFT JOIN crm_target.alfcc_acciones b
    ON a.ll_accion = b.id
WHERE a.ll_base = 'mayo 2026'
  AND a.ll_fecha = '2026-05-11'
LIMIT 10
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [31]:
df_agente.head()

,cid,phone_number,fecha_llamada,duracion,nombre,id_banco,codigo,hora_llamada,tramo,fecha_agenda,campana
0,3541336,977532775,2026-05-11,95,SOLICITÓ NO SER CONTACTADO,17,ACC017,2026-05-11 09:00:47,9,None,ALFCC
1,3503484,984223175,2026-05-11,110,CORTA LLAMADA,22,ACC022,2026-05-11 09:00:42,9,None,ALFCC
2,3450707,963325036,2026-05-11,12,SOLICITÓ NO SER CONTACTADO,17,ACC017,2026-05-11 09:02:51,9,None,ALFCC
3,3554739,936567373,2026-05-11,56,SOLICITÓ NO SER CONTACTADO,17,ACC017,2026-05-11 09:02:38,9,None,ALFCC
4,3582742,927981215,2026-05-11,57,CORTA LLAMADA,22,ACC022,2026-05-11 09:03:29,9,None,ALFCC


## actualizar retiro telef 

In [ ]:

cols_tel = [
    'cl_telf1','cl_telf2','cl_telf3','cl_telf4','cl_telf5',
    'cl_telf6','cl_telf7','cl_telf8','cl_telf9','cl_telf10',
    'cl_movil','cl_celular','cl_telefono'
]

df_long = df_dni.melt(
    id_vars='NUMERO_DOCUMENTO',
    value_vars=cols_tel,
    var_name='tipo_telf',
    value_name='CELULAR'
)
df_long['CELULAR'] = (
    df_long['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_long = df_long[
    (df_long['CELULAR'].notna()) &
    (df_long['CELULAR'] != '') &
    (df_long['CELULAR'].str.len() == 9) &
    (df_long['CELULAR'].str.startswith('9'))
]



In [3]:
filename='blacklist_celulares.txt'
filePath = os.path.join(ruta_csv, filename)

df_list = pd.read_csv(filePath)

In [4]:

print(f"df_dni filas: {df_long.shape[0]}")
print(f"df_list filas: {df_list.shape[0]}")
df_list['CELULAR'] = (
    df_list['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)

df_dni filas: 171922
df_list filas: 607337


In [ ]:
# dnis = [
#     "42626543",
#     "18070214",
#     "40503275",
#     "46866131",
#     "70812815",
#     "46868831",
#     "73037069"
# ]

# telefonos = [
#     "979600075",
#     "969226378",
#     "991509057",
#     "960814290",
#     "965786244",
#     "951345731",
#     "950931938",
#     "996612357"
# ]

# import pandas as pd

# df_manual_dni = pd.DataFrame({
#     "dni_cliente": dnis,
# })


# df_manual_telf = pd.DataFrame({
#     "CELULAR": telefonos
# })


In [ ]:
# data = [
#     ("42626543", "979600075"),
#     ("18070214", "969226378"),
#     ("40503275", "991509057"),
#     ("46866131", "960814290"),
#     ("70812815", "965786244"),
#     ("46868831", "951345731"),
#     ("95093138", "950931938"),
#     ("73037069", "996612357")
# ]

# df_manual = pd.DataFrame(data, columns=["DNI", "TELEFONO"])

In [ ]:
spark_df = spark.createDataFrame(df_manual)
spark_df.show()


In [38]:
df_long.head()

,NUMERO_DOCUMENTO,tipo_telf,CELULAR
2,80708936,cl_telf1,931185554
4,80686119,cl_telf1,969443669
6,80681915,cl_telf1,980257187
7,80681241,cl_telf1,973476485
8,80678740,cl_telf1,900212933


In [5]:
df_list['CELULAR'] = df_list['CELULAR'].astype(str)

df_list = df_list.merge(
    df_long,
    on="CELULAR",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 169


In [7]:
df_list['cl_estado']='3'
df_list['estado']='Retirar Telf'
df_list=df_list[['NUMERO_DOCUMENTO','cl_estado','estado']]
df_list.count()

NUMERO_DOCUMENTO    169
cl_estado           169
estado              169
dtype: int64

In [8]:
df_list = df_list.drop_duplicates(
    subset=['NUMERO_DOCUMENTO'],
    keep='last'
)
df_list.count()

NUMERO_DOCUMENTO    85
cl_estado           85
estado              85
dtype: int64

In [9]:
df_list.head()

,NUMERO_DOCUMENTO,cl_estado,estado
1,44115550,3,Retirar Telf
3,41247835,3,Retirar Telf
5,42979934,3,Retirar Telf
7,10282742,3,Retirar Telf
9,75382244,3,Retirar Telf


In [10]:
reemplazo = {
    'NUMERO_DOCUMENTO': 'col_01',
    'estado': 'col_02',
    'cl_estado': 'col_03'
}
df_list = df_list.rename(columns=reemplazo)

In [10]:
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [16]:

df_list.to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

85

In [17]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfcc_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO = b.col_01 
SET 
    a.cl_estado = b.col_03,
    a.estado = b.col_02
WHERE 
    a.cl_base = 'mayo 2026'
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas afectadas:", result.rowcount)

Filas afectadas: 85


In [18]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 85


In [ ]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfcc_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO COLLATE utf8mb4_general_ci
       = b.col_01 COLLATE utf8mb4_general_ci
SET 
    a.cl_estado = b.col_02,
    a.estado = b.col_03
WHERE 
    a.cl_base = 'mayo 2026'
    AND (
        IFNULL(a.cl_estado,'') COLLATE utf8mb4_general_ci 
            <> IFNULL(b.col_02,'') COLLATE utf8mb4_general_ci
        OR 
        IFNULL(a.estado,'') COLLATE utf8mb4_general_ci 
            <> IFNULL(b.col_03,'') COLLATE utf8mb4_general_ci
    )
LIMIT 1000
"""

with engine.begin() as conn:
    total = 0

    while True:
        result = conn.execute(text(query))
        filas = result.rowcount
        total += filas

        print("Filas afectadas:", filas)

        if filas == 0:
            break

    print("TOTAL ACTUALIZADO:", total)

OperationalError: (pymysql.err.OperationalError) (1969, 'Query execution was interrupted (max_statement_time exceeded)')
[SQL: 
UPDATE crm_target.alfcc_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO COLLATE utf8mb4_general_ci
       = b.col_01 COLLATE utf8mb4_general_ci
SET 
    a.cl_estado = b.col_02,
    a.estado = b.col_03
WHERE 
    a.cl_base = 'mayo 2026'
    AND (
        IFNULL(a.cl_estado,'') COLLATE utf8mb4_general_ci 
            <> IFNULL(b.col_02,'') COLLATE utf8mb4_general_ci
        OR 
        IFNULL(a.estado,'') COLLATE utf8mb4_general_ci 
            <> IFNULL(b.col_03,'') COLLATE utf8mb4_general_ci
    )
LIMIT 1000
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:

engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
* FROM tb_temporal
"""

df_prueba_01 = pd.read_sql(query, engine_mysql)

df_prueba_01.columns

Index(['col_01', 'col_02', 'col_03', 'col_04'], dtype='object')

In [8]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfcc_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="cl_estado",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 11788
Lote 0 - 2000 actualizado | filas afectadas: 1927
Lote 2000 - 4000 actualizado | filas afectadas: 1931
Lote 4000 - 6000 actualizado | filas afectadas: 1908
Lote 6000 - 8000 actualizado | filas afectadas: 1898
Lote 8000 - 10000 actualizado | filas afectadas: 1897
Lote 10000 - 11788 actualizado | filas afectadas: 1690
Proceso terminado. Total filas afectadas: 11251


## actualizar retiro dni

In [25]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
DISTINCT 
NUMERO_DOCUMENTO as dni_cliente
FROM alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["dni_cliente"] = (
    df_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)


In [26]:
filename='blacklist_dni.txt'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)
df_list.head(2)

,DNI
0,0
1,14


In [30]:

df_list.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
df_list["dni_cliente"] = (
    df_list["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_list.columns)
print(df_dni.columns)


Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [28]:
print(df_dni.columns)
print(df_list.columns)

Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [31]:
df_list = df_list.merge(
    df_dni,
    on="dni_cliente",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 0


In [19]:
df_list['retiro']='Retirar BlackList'
df_list=df_list[['dni_cliente','retiro']]
df_list.count()

dni_cliente    1648
retiro         1648
dtype: int64

In [ ]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="estado",
    col_llave_df="dni_cliente",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1648
Lote 0 - 1648 actualizado | filas afectadas: 0
Proceso terminado. Total filas afectadas: 0


## ejecutar query

In [ ]:
from sqlalchemy import create_engine


engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)
with engine.begin() as conn:
    result = conn.execute(text("""
        UPDATE crm_target.alfcc_clientes
        SET cl_estado = 3
        WHERE estado <> 'ACTIVO'
        AND cl_base = 'mayo 2026'
    """))
    
    print("Filas afectadas:", result.rowcount)

## actualizar dni a otro lote

In [3]:
filename='dni_repetidos_alfin.csv'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

In [5]:
df_list["NUMERO_DOCUMENTO"] = (
    df_list["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)
df_list.head()

,NUMERO_DOCUMENTO
0,46507674
1,40963267
2,41782588
3,40658062
4,09457398


In [6]:
df_list['lote_ref']='BD-Target -ASM'

In [7]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="lote",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="lote_ref",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1066
Lote 0 - 1066 actualizado | filas afectadas: 1066
Proceso terminado. Total filas afectadas: 1066


In [ ]:
BD-Target -ASM

In [5]:
exec_query_sql(server_zeus, "MAEBA", user_zeus, pwd_zeus, "ADM_OBJ_TG.spFunnelDinersTc", "SP funnel diners_tc Zeus")

SP funnel diners_tc Zeus | realizado | duración: 19.96 seg


In [ ]:
overwrite_table_SQL(spark,df_prueba_1,f'borrar_TARGET_202604_01',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_2,f'borrar_TARGET_202604_02',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_ch,f'borrar_TARGET_202604_ch',server_kishin,user_kishin,pwd_kishin,'DANTALION')


df_list filas: 1066


In [ ]:
df_dni filas: 128183
df_list filas: 12546

In [ ]:
ssss

In [7]:
print(df_list.columns)
print(df_long.columns)

Index(['TELEFONO'], dtype='object')
Index(['NUMERO_DOCUMENTO', 'tipo_telf', 'TELEFONO'], dtype='object')


df_list filas: 1


In [ ]:
df_list filas: 2036

NUMERO_DOCUMENTO    1
retiro              1
dtype: int64

In [10]:
df_list.head()

,NUMERO_DOCUMENTO,retiro
0,40503275,Retirar Telef


Total registros a procesar: 1
Lote 0 - 1 actualizado | filas afectadas: 1
Proceso terminado. Total filas afectadas: 1
